<a href="https://colab.research.google.com/github/vadimdddd/CV_task_contour_detection_cad/blob/main/CV_Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Получаем датасет с гугл диска


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

!cp -r /content/gdrive/MyDrive/alpha_version/ .

Установка необходимых модулей для работы


In [ ]:
!pip install --upgrade opencv-python-headless pymupdf numpy matplotlib pillow tqdm gradio==6.22.0 --quiet

Импортируем модули

In [ ]:
import os
import cv2
import time
import fitz
import json
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from pathlib import Path

Для сохранения памяти просто импортируем уже папку с png и тогда следующий фрагмент кода можно пропустить

---



In [ ]:
!cp -r /content/gdrive/MyDrive/alpha_version_png/ .

Преобразуем pdf в png для лучшего качества изображений

In [ ]:
def process_pdfs_colab():
    """
    Пакетная конвертация PDF-чертежей в PNG-изображения с автоматическим обрезанием по ориентации страницы.

    === ВХОДНЫЕ ПАРАМЕТРЫ ===
    Функция не принимает аргументов. Все параметры жестко заданы внутри:

    Параметры путей:
        base_dir (Path): "/content/gdrive/MyDrive" - базовая директория в Google Drive
        source_dir (Path): base_dir / "alpha_version" - папка с исходными PDF-файлами
        dest_dir (Path): base_dir / "alpha_version_png" - папка для сохранения PNG

    Параметры ожидания (для Google Colab):
        max_wait_time (int): 30 сек - максимальное время ожидания появления папки
        wait_interval (int): 2 сек - интервал между проверками

    Параметры обрезки страницы (clip_rect):
        vertical_rect (fitz.Rect): Rect(0, 0, 800, 1000) - для вертикальных чертежей
        horizontal_rect (fitz.Rect): Rect(0, 0, 1800, 1300) - для горизонтальных чертежей
        page_width > page_height: условие определения ориентации

    Параметры конвертации PDF -> PNG:
        zoom (float): 600 / 72 ≈ 8.33 - коэффициент масштабирования (итоговое разрешение ~600 DPI)
        mat (fitz.Matrix): Matrix(zoom, zoom) - матрица трансформации
        alpha (bool): False - отключение альфа-канала (без прозрачности)

    Параметры сохранения PNG:
        optimize (bool): True - оптимизация размера файла
        compress_level (int): 0 - уровень сжатия (0 = без сжатия, макс. качество)

    Параметры именования:
        new_base_name (str): f"sample_{idx}" - базовое имя с индексом
        png_filename (str): f"{new_base_name}.png" - имя PNG-файла
        txt_filename (str): f"{new_base_name}.txt" - имя TXT-файла с метаданными

    === ВОЗВРАЩАЕМЫЕ ЗНАЧЕНИЯ ===
    Returns:
        List[PIL.Image] | None:
            Список загруженных изображений в формате PIL.
            Возвращает None, если папка не найдена или нет PDF-файлов.

    === ПОБОЧНЫЕ ЭФФЕКТЫ (СОЗДАЁТ ФАЙЛЫ НА ДИСКЕ) ===
        - PNG-файлы: {dest_dir}/sample_{idx}.png
        - TXT-файлы: {dest_dir}/sample_{idx}.txt (содержат original_name и new_name)

    === АЛГОРИТМ РАБОТЫ ===
        1. Проверка наличия папки /content/gdrive/MyDrive/alpha_version
           - Если нет → ждет до max_wait_time секунд
           - Если так и нет → ошибка и return None
        2. Создание папки назначения /content/gdrive/MyDrive/alpha_version_png
        3. Поиск всех PDF-файлов в папке источника
        4. Для каждого PDF-файла:
           - Открыть PDF
           - Загрузить первую страницу
           - Определить ориентацию:
             * Если ширина > высота → горизонтальный (1800×1300)
             * Иначе → вертикальный (800×1000)
           - Применить масштаб 600 DPI (zoom = 600/72)
           - Сконвертировать в PNG
           - Сохранить PNG в папку назначения
           - Создать TXT с метаданными
           - Добавить изображение в список
        5. Показать первые 5 изображений

    === ВЛИЯНИЕ ПАРАМЕТРОВ ===
        Увеличение zoom:
            + Более высокое разрешение (качество)
            - Больший размер файла
            - Медленнее обработка

        Уменьшение zoom:
            + Меньший размер файла
            + Быстрее обработка
            - Более низкое разрешение (качество)

        Увеличение rect (области обрезки):
            + Захватывает больше области чертежа
            - Может захватывать лишнее (поля, рамки)

        Уменьшение rect:
            + Обрезает лишнее (поля, рамки)
            - Может обрезать часть чертежа

        compress_level=0:
            + Максимальное качество
            - Большой размер файла

        compress_level=9:
            + Маленький размер файла
            - Потеря качества (сжатие с потерями)

    === ОСОБЕННОСТИ ===
        - Обрабатывается только первая страница PDF
        - Все PNG имеют одинаковое разрешение (~600 DPI)
        - Размеры обрезки фиксированы (800×1000 или 1800×1300)
        - PNG сохраняются без сжатия (компрессия = 0)
        - Оптимизировано для работы в Google Colab с Google Drive

    === ВОЗМОЖНЫЕ ОШИБКИ ===
        - Папка alpha_version не найдена → return None
        - Нет PDF-файлов в папке → return None
        - Поврежденный PDF-файл → пропускает файл, продолжает обработку
        - DecompressionBombWarning → изображение слишком большое

    === ПРИМЕР ИСПОЛЬЗОВАНИЯ ===
        # Запуск конвертации
        images = process_pdfs_colab()

        # Получение первого изображения
        if images:
            first_image = images[0]  # PIL Image

        # Получение путей к файлам
        png_files = sorted(Path("/content/gdrive/MyDrive/alpha_version_png").glob("*.png"))
    """
    base_dir = Path("/content/gdrive/MyDrive")
    source_dir = base_dir / "alpha_version"
    dest_dir = base_dir / "alpha_version_png"

    print(f"Текущая директория: {Path.cwd()}")
    print(f"Ищем PDF здесь: {source_dir.resolve()}")
    print(f"Сохраним PNG здесь: {dest_dir.resolve()}\n")

    max_wait_time = 30  # Секунды: максимальное время ожидания папки
    wait_interval = 2   # Секунды: интервал между проверками
    elapsed = 0         # Секунды: накопленное время ожидания

    # Ожидание появления папки с PDF (для Google Colab)
    while not source_dir.exists() and elapsed < max_wait_time:
        if elapsed == 0:
            print("Ожидаем появления папки alpha_version...")
        time.sleep(wait_interval)
        elapsed += wait_interval

    if not source_dir.exists():
        print("\nКРИТИЧЕСКАЯ ОШИБКА: Папка /content/gdrive/MyDrive/alpha_version не найдена.")
        return

    # Создание папки для PNG (если её нет)
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Список всех PDF-файлов в папке источника
    pdf_files = sorted(source_dir.glob("*.pdf"))

    if not pdf_files:
        print(f"В папке {source_dir} не найдено PDF файлов.")
        return

    saved_images = []  # Список для хранения PIL Image объектов

    # Обработка каждого PDF
    for idx in tqdm(range(len(pdf_files)), desc="Обработка чертежей", unit="файл", ncols=100):
        pdf_path = pdf_files[idx]
        new_base_name = f"sample_{idx}"  # Имя файла: sample_0, sample_1, ...

        try:
            # === 1. ОТКРЫТИЕ PDF ===
            doc = fitz.open(str(pdf_path))
            page = doc.load_page(0)  # Только первая страница

            # === 2. ОПРЕДЕЛЕНИЕ ОРИЕНТАЦИИ ===
            page_width = page.rect.width
            page_height = page.rect.height

            # Размеры обрезки в зависимости от ориентации
            vertical_rect = fitz.Rect(0, 0, 800, 1000)      # Для вертикальных (книжная)
            horizontal_rect = fitz.Rect(0, 0, 1800, 1300)   # Для горизонтальных (альбомная)

            # Выбор области обрезки по ориентации
            clip_rect = horizontal_rect if page_width > page_height else vertical_rect

            # === 3. КОНВЕРТАЦИЯ В PNG ===
            # zoom = 600/72 ≈ 8.33 → итоговое разрешение ~600 DPI
            zoom = 600 / 72
            mat = fitz.Matrix(zoom, zoom)

            # Рендеринг страницы в изображение
            pix = page.get_pixmap(matrix=mat, clip=clip_rect, alpha=False)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

            # === 4. СОХРАНЕНИЕ ФАЙЛОВ ===
            png_filename = f"{new_base_name}.png"
            txt_filename = f"{new_base_name}.txt"

            png_path = dest_dir / png_filename
            txt_path = dest_dir / txt_filename

            # Сохранение PNG (без сжатия для максимального качества)
            img.save(png_path, "PNG", optimize=True, compress_level=0)

            # Сохранение TXT с метаданными
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(f"original_name: {pdf_path.name}\n")
                f.write(f"new_name: {png_filename}\n")

            doc.close()
            saved_images.append(img)

        except Exception as e:
            print(f"\nОшибка в файле {pdf_path.name}: {type(e).__name__} - {e}")
            continue

    print(f"\nГотово. Файлы сохранены в: {dest_dir.resolve()}")
    return saved_images


# === ВЫЗОВ ФУНКЦИИ ===
images_list = process_pdfs_colab()

# === ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ ===
if images_list:
    # Показываем первые 5 изображений
    fig, axes = plt.subplots(nrows=1, ncols=min(5, len(images_list)), figsize=(20, 8))
    if len(images_list) == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i < len(images_list):
            ax.imshow(images_list[i])
            ax.set_title(f'sample_{i}', fontsize=10)
            ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Список изображений пуст или возникла ошибка при обработке.")

Выделим регион на изображении, для которого нужно найти контур

In [ ]:
DATA_DIR = Path("/content/gdrive/MyDrive/alpha_version_png")
OUTPUT_JSON = Path("/content/gdrive/MyDrive/roi_data.json")

def auto_detect_roi(image_path):
    """
    Автоматическое определение области интереса (ROI) на чертеже.
    Находит рамку чертежа и определяет внутреннюю область для дальнейшей обработки.

    === ВХОДНЫЕ ПАРАМЕТРЫ ===
    Args:
        image_path (str | Path): Путь к PNG-изображению чертежа

    === ВОЗВРАЩАЕМЫЕ ЗНАЧЕНИЯ ===
    Returns:
        list | None:
            Список координат ROI в формате [x1, y1, x2, y2], где:
            - x1, y1 - верхний левый угол
            - x2, y2 - нижний правый угол
            Возвращает None, если изображение не загружено или контуры не найдены

    === ВНУТРЕННИЕ ПАРАМЕТРЫ ===
    Параметры бинаризации:
        THRESH_BINARY_INV + THRESH_OTSU: инвертированная бинаризация с методом Оцу
        - Автоматически определяет оптимальный порог
        - Инвертирует результат (фон становится белым, объекты - черными)

    Параметры морфологии:
        kernel (tuple): (15, 15) - размер ядра для морфологического закрытия
        MORPH_CLOSE: операция закрытия (дилатация → эрозия)
        iterations (int): 2 - количество повторений операции
        - Закрывает разрывы в контурах
        - Соединяет близкие объекты
        - Удаляет мелкие отверстия внутри объектов

        Влияние изменения kernel:
            Увеличение (>15):
                + Сильнее соединяет объекты
                + Лучше закрывает большие разрывы
                - Может склеивать разные объекты
                - Потеря мелких деталей

            Уменьшение (<15):
                + Сохраняет мелкие детали
                + Меньше склеивает объекты
                - Может не закрыть разрывы
                - Оставляет мелкие отверстия

    Параметры фильтрации контуров:
        min_area (int): 5000 - минимальная площадь контура для кандидата в ROI
        - Отсеивает мелкие объекты (шум, артефакты)

        Влияние изменения:
            Увеличение (>5000):
                + Меньше шума
                - Может пропустить маленькую рамку
            Уменьшение (<5000):
                + Находит маленькие рамки
                - Больше шума и мусора

    Параметры отступов (если ROI не найден):
        margin_w (float): w_bg * 0.1 - отступ по ширине (10% от ширины рамки)
        margin_h (float): h_bg * 0.1 - отступ по высоте (10% от высоты рамки)
        - Используется для создания ROI с отступом от краев рамки

        Влияние изменения:
            Увеличение (>0.1):
                + Больше отступ от краев
                - Меньше область обработки
            Уменьшение (<0.1):
                + Больше область обработки
                - Ближе к краям рамки

    === АЛГОРИТМ РАБОТЫ ===
        1. Загрузка изображения
        2. Преобразование в оттенки серого
        3. Бинаризация методом Оцу (инвертированная)
        4. Морфологическое закрытие для соединения контуров
        5. Поиск всех внешних контуров
        6. Сортировка контуров по площади (от большего к меньшему)
        7. Нахождение самого большого контура (это рамка чертежа)
        8. Поиск внутреннего контура (область внутри рамки):
           - Проверка: x > x_bg и y > y_bg
           - Проверка: (x+w) < (x_bg+w_bg) и (y+h) < (y_bg+h_bg)
           - Т.е. контур должен находиться строго внутри рамки
        9. Если внутренний контур найден → возвращаем его координаты
        10. Если не найден → создаем ROI с отступами 10% от краев рамки

    === ПРИМЕР ВОЗВРАЩАЕМЫХ КООРДИНАТ ===
        [100, 150, 1800, 1200] где:
        - x1=100, y1=150 (верхний левый угол)
        - x2=1800, y2=1200 (нижний правый угол)

    === ВОЗМОЖНЫЕ ОШИБКИ ===
        - Изображение не загружено → return None
        - Контуры не найдены → return None
        - Рамка не найдена → return None (через отсутствие contours)
    """
    # Загрузка изображения
    img = cv2.imread(str(image_path))
    if img is None:
        return None

    # Преобразование в оттенки серого
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Бинаризация методом Оцу (инвертированная)
    # THRESH_BINARY_INV: объекты становятся белыми, фон - черным
    # THRESH_OTSU: автоматический подбор порога
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Морфологическое закрытие: соединяет разорванные линии
    # kernel (15,15): большое ядро для соединения крупных объектов
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Поиск внешних контуров
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    # Сортировка контуров по площади (от большего к меньшему)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    # Самый большой контур = рамка чертежа
    x_bg, y_bg, w_bg, h_bg = cv2.boundingRect(contours[0])

    roi_found = None

    # Поиск внутреннего контура (область внутри рамки)
    for cnt in contours[1:]:  # Пропускаем самый большой (рамку)
        area = cv2.contourArea(cnt)
        if area < 5000:  # Минимальная площадь для кандидата
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        # Проверяем, что контур находится строго внутри рамки
        if (x > x_bg and y > y_bg and
            (x+w) < (x_bg+w_bg) and (y+h) < (y_bg+h_bg)):
            # Нашли внутреннюю область
            roi_found = [x, y, x+w, y+h]
            break

    # Если внутренний контур не найден, создаем ROI с отступами
    if not roi_found:
        # Отступы 10% от размера рамки
        margin_w, margin_h = int(w_bg * 0.1), int(h_bg * 0.1)
        roi_found = [
            x_bg + margin_w,           # x1 - левый верхний угол
            y_bg + margin_h,           # y1 - левый верхний угол
            x_bg + w_bg - margin_w,    # x2 - правый нижний угол
            y_bg + h_bg - margin_h     # y2 - правый нижний угол
        ]

    return roi_found


# === ОСНОВНОЙ БЛОК: АВТОМАТИЧЕСКАЯ РАЗМЕТКА ROI ===
print("Начинаю автоматический поиск областей интереса...")

# Загрузка существующего словаря ROI из JSON (если есть)
roi_dict = {}
if OUTPUT_JSON.exists():
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        try:
            roi_dict = json.load(f)
        except Exception:
            roi_dict = {}

# Получение списка всех PNG-файлов
png_files = sorted(DATA_DIR.glob("*.png"))
processed_count = 0

# Обработка каждого файла
for img_path in tqdm(png_files, desc="Авто-разметка"):
    filename = img_path.name

    # Пропускаем уже обработанные файлы
    if filename in roi_dict:
        processed_count += 1
        continue

    # Автоматическое определение ROI
    coords = auto_detect_roi(img_path)

    if coords:
        roi_dict[filename] = coords
        processed_count += 1

# Сохранение результатов в JSON
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(roi_dict, f, indent=4, ensure_ascii=False)

print(f"\nГотово! Размечено {processed_count} новых файлов.")
print(f"Всего в базе: {len(roi_dict)} записей.")

Логика поиска контуров

In [ ]:
# ============================================================
# КОНСТАНТЫ ДЛЯ НАСТРОЙКИ АЛГОРИТМА
# ============================================================

# Параметры размеров контуров
MIN_AREA_PERCENT = 0.0005     # Минимальная площадь контура
MAX_AREA_PERCENT = 0.20       # Максимальная площадь контура

# Параметры бинаризации
BINARY_THRESHOLD = 120        # Порог бинаризации
ADAPTIVE_BLOCK_SIZE = 3       # Размер блока адаптивного порога
ADAPTIVE_C_CONST = 2          # Константа адаптивного порога

# Параметры морфологии
MORPH_KERNEL_SIZE = (3, 3)    # Размер ядра морфологии
MORPH_ITERATIONS = 3          # Количество итераций морфологии

# Параметры фильтрации контуров
FILL_RATIO_THRESHOLD = 0.25   # Минимальная заполненность контура
EPSILON_FACTOR = 0.005        # Точность аппроксимации
MIN_POINTS = 6                # Минимальное количество точек в контуре

# Параметры визуализации
LINE_THICKNESS = 10           # Толщина линии контура

# Параметры удаления штампа
FRAME_MIN_AREA_RATIO = 0.3    # Минимальная площадь рамки
STAMP_MARGIN_RATIO = 0.02     # Отступ от рамки

# Специальные размеры штампа для крупного чертежа (w=14036, h=9930)
SPECIAL_STAMP_SIZES = [
    (0.10, 0.08),
    (0.08, 0.06),
    (0.12, 0.10),
    (0.15, 0.12),
]

# Стандартные размеры штампа
STANDARD_STAMP_SIZES = [
    (0.20, 0.15),
    (0.15, 0.12),
    (0.12, 0.10),
    (0.25, 0.18),
    (0.10, 0.08),
    (0.30, 0.20),
]

# ============================================================


def extract_part_contours(image_array, filename):
    try:
        img_bgr = cv2.cvtColor(np.array(image_array), cv2.COLOR_RGB2BGR)
    except Exception:
        img_bgr = np.array(image_array).astype(np.uint8)

    h, w = img_bgr.shape[:2]
    img_original = img_bgr.copy()

    is_horizontal = w > h
    print(f"Ориентация чертежа: {'ГОРИЗОНТАЛЬНЫЙ' if is_horizontal else 'ВЕРТИКАЛЬНЫЙ'}")

    total_area_px = h * w
    min_area = int(total_area_px * MIN_AREA_PERCENT)
    max_area = int(total_area_px * MAX_AREA_PERCENT)

    shift_x, shift_y = 0, 0

    # === УДАЛЕНИЕ ШТАМПА (только для горизонтальных чертежей) ===
    if is_horizontal:
        gray_full = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        _, binary_full = cv2.threshold(gray_full, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        contours_full, _ = cv2.findContours(binary_full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours_full:
            largest_contour = max(contours_full, key=cv2.contourArea)
            x_frame, y_frame, w_frame, h_frame = cv2.boundingRect(largest_contour)
            frame_area = w_frame * h_frame

            if frame_area > total_area_px * FRAME_MIN_AREA_RATIO:
                print(f"Обнаружена рамка: x={x_frame}, y={y_frame}, w={w_frame}, h={h_frame}")

                # Определяем размеры штампа
                if w_frame == 14036 and h_frame == 9930:
                    print("Обнаружен крупный чертеж, применяем специальные размеры штампа")
                    stamp_sizes = SPECIAL_STAMP_SIZES
                    min_content, max_content = 0.03, 0.80
                    extra_margin = 0.1
                else:
                    print("Используем стандартные размеры штампа")
                    stamp_sizes = STANDARD_STAMP_SIZES
                    min_content, max_content = 0.05, 0.70
                    extra_margin = 0.05

                stamp_removed = False

                for width_ratio, height_ratio in stamp_sizes:
                    stamp_width = int(w_frame * width_ratio)
                    stamp_height = int(h_frame * height_ratio)
                    stamp_x = x_frame + w_frame - stamp_width
                    stamp_y = y_frame + h_frame - stamp_height

                    if stamp_x < x_frame or stamp_y < y_frame:
                        continue

                    print(f"Пробуем размер штампа: {width_ratio:.2f}x{height_ratio:.2f}")

                    test_roi = img_bgr[stamp_y:stamp_y + stamp_height, stamp_x:stamp_x + stamp_width]
                    if test_roi.size == 0:
                        continue

                    gray_test = cv2.cvtColor(test_roi, cv2.COLOR_BGR2GRAY)
                    _, binary_test = cv2.threshold(gray_test, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

                    non_zero = cv2.countNonZero(binary_test)
                    total_pixels = stamp_width * stamp_height
                    content_ratio = non_zero / total_pixels if total_pixels > 0 else 0

                    print(f"  Содержимое: {content_ratio:.2%}")

                    if min_content < content_ratio < max_content:
                        print(f"✓ НАЙДЕН ШТАМП! Размер {width_ratio:.2f}x{height_ratio:.2f}")

                        cv2.rectangle(img_bgr, (stamp_x, stamp_y),
                                      (stamp_x + stamp_width, stamp_y + stamp_height),
                                      (255, 255, 255), -1)

                        extra_margin_w = int(stamp_width * extra_margin)
                        extra_margin_h = int(stamp_height * extra_margin)

                        cv2.rectangle(img_bgr,
                                      (stamp_x + stamp_width, stamp_y),
                                      (stamp_x + stamp_width + extra_margin_w, stamp_y + stamp_height),
                                      (255, 255, 255), -1)

                        cv2.rectangle(img_bgr,
                                      (stamp_x, stamp_y + stamp_height),
                                      (stamp_x + stamp_width, stamp_y + stamp_height + extra_margin_h),
                                      (255, 255, 255), -1)

                        stamp_removed = True
                        break

                if not stamp_removed:
                    print("Не удалось найти штамп, используем стандартный размер")
                    stamp_width = int(w_frame * 0.20)
                    stamp_height = int(h_frame * 0.15)
                    stamp_x = x_frame + w_frame - stamp_width
                    stamp_y = y_frame + h_frame - stamp_height
                    cv2.rectangle(img_bgr, (stamp_x, stamp_y),
                                  (stamp_x + stamp_width, stamp_y + stamp_height),
                                  (255, 255, 255), -1)

                # Обрезка по рамке
                margin_x = int(w_frame * STAMP_MARGIN_RATIO) + x_frame
                margin_y = int(h_frame * STAMP_MARGIN_RATIO) + y_frame
                crop_x2 = x_frame + w_frame - int(w_frame * STAMP_MARGIN_RATIO)
                crop_y2 = y_frame + h_frame - int(h_frame * STAMP_MARGIN_RATIO)

                img_bgr = img_bgr[margin_y:crop_y2, margin_x:crop_x2]
                shift_x, shift_y = margin_x, margin_y

    # Если штамп не удален или чертеж вертикальный - стандартная обрезка
    if shift_x == 0 and shift_y == 0:
        margin_x = int(w * 0.05)
        margin_y = int(h * 0.05)
        img_bgr = img_bgr[margin_y:h-margin_y, margin_x:w-margin_x]
        shift_x, shift_y = margin_x, margin_y

    # === ОСНОВНАЯ ОБРАБОТКА ===
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    _, binary = cv2.threshold(enhanced, BINARY_THRESHOLD, 255, cv2.THRESH_BINARY_INV)

    binary_adaptive = cv2.adaptiveThreshold(
        enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C_CONST
    )

    binary = cv2.bitwise_or(binary, binary_adaptive)

    kernel_thin = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, MORPH_KERNEL_SIZE)
    thinned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_thin, iterations=MORPH_ITERATIONS)

    contours, _ = cv2.findContours(thinned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    output_vis = img_original.copy()

    if not contours:
        return Image.fromarray(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)), \
               Image.fromarray(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))

    sorted_cnts = sorted(contours, key=cv2.contourArea, reverse=True)

    final_polylines = []

    for cnt in sorted_cnts:
        area = cv2.contourArea(cnt)

        if area < min_area:
            break
        if area > max_area:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)
        rect_area = cw * ch

        fill_ratio = area / rect_area
        if fill_ratio < FILL_RATIO_THRESHOLD:
            continue

        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, EPSILON_FACTOR * peri, True)

        if len(approx) < MIN_POINTS:
            continue

        shifted_approx = approx + np.array([shift_x, shift_y])
        final_polylines.append(shifted_approx)

    if final_polylines:
        cv2.polylines(output_vis, final_polylines, isClosed=True,
                      color=(0, 255, 0), thickness=LINE_THICKNESS)

    original_out = Image.fromarray(cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB))
    result_out = Image.fromarray(cv2.cvtColor(output_vis, cv2.COLOR_BGR2RGB))
    return original_out, result_out

Сохранение результатов обработки

In [ ]:
OUTPUT_DIR = Path("/content/gdrive/MyDrive/results")

def create_comparison_image(original_img, processed_img, filename):
    """
    Создает изображение-пару: оригинал слева, результат справа.
    """
    target_h = 800

    orig_w, orig_h = original_img.size
    proc_w, proc_h = processed_img.size

    scale_orig = target_h / orig_h
    scale_proc = target_h / proc_h

    new_orig_w = int(orig_w * scale_orig)
    new_proc_w = int(proc_w * scale_proc)

    original_resized = original_img.resize((new_orig_w, target_h), Image.Resampling.LANCZOS)
    processed_resized = processed_img.resize((new_proc_w, target_h), Image.Resampling.LANCZOS)

    total_w = new_orig_w + new_proc_w
    comparison_img = Image.new('RGB', (total_w, target_h), color='white')

    comparison_img.paste(original_resized, (0, 0))
    comparison_img.paste(processed_resized, (new_orig_w, 0))

    return comparison_img

if not DATA_DIR.exists():
    print(f"Папка с входными данными не найдена: {DATA_DIR}")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    png_files = sorted(DATA_DIR.glob("*.png"))

    if not png_files:
        print("В папке alpha_version_png не найдено PNG-файлов.")
    else:
        print(f"Найдено файлов: {len(png_files)}")
        print("Начинаю обработку с улучшенным алгоритмом...")
        success_count = 0
        failed_files = []

        for img_path in tqdm(png_files, desc="Пакетная обработка"):
            try:
                pil_image = Image.open(img_path).convert("RGB")

                original_out, result_out = extract_part_contours(pil_image, img_path.name)

                final_image = create_comparison_image(original_out, result_out, img_path.name)

                save_path = OUTPUT_DIR / f"result_{img_path.name}"
                final_image.save(save_path, format="PNG", optimize=True)
                success_count += 1

            except Exception as e:
                print(f"\nОшибка обработки {img_path.name}: {e}")
                failed_files.append(img_path.name)
                continue

        print(f"\nГотово! Обработано успешно: {success_count}/{len(png_files)}")
        if failed_files:
            print(f"Не удалось обработать: {', '.join(failed_files)}")
        print(f"Результаты лежат здесь: {OUTPUT_DIR}")

Интерфейс демонстрации

In [ ]:
TEMP_PDF_RENDER = "/tmp/gradio_pdf_render.png"

def render_pdf_to_image(pdf_file_path):
    """
    Конвертирует первую страницу PDF в изображение PNG.

    === ВХОДНЫЕ ПАРАМЕТРЫ ===
    Args:
        pdf_file_path (str | Path): Путь к PDF-файлу

    === ВОЗВРАЩАЕМЫЕ ЗНАЧЕНИЯ ===
    Returns:
        PIL.Image | None: Изображение первой страницы PDF в формате RGB или None при ошибке

    === ВНУТРЕННИЕ ПАРАМЕТРЫ ===
        zoom_x, zoom_y (float): 300 / 72 ≈ 4.17 - разрешение ~300 DPI
            - Увеличение: выше качество, больше размер
            - Уменьшение: ниже качество, меньше размер

        alpha (bool): False - отключение альфа-канала

        TEMP_PDF_RENDER (str): "/tmp/gradio_pdf_render.png" - временный файл

    === ОСОБЕННОСТИ ===
        - Обрабатывает только первую страницу PDF
        - Сохраняет во временный файл, затем загружает как PIL Image
        - Использует разрешение 300 DPI
    """
    try:
        doc = fitz.open(str(pdf_file_path))
        if doc.page_count == 0:
            return None
        page = doc.load_page(0)  # Только первая страница
        zoom_x = zoom_y = 300 / 72  # ~300 DPI
        mat = fitz.Matrix(zoom_x, zoom_y)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        pix.save(TEMP_PDF_RENDER)
        return Image.open(TEMP_PDF_RENDER).convert("RGB")
    except Exception as e:
        print(f"Ошибка чтения PDF: {e}")
        return None


def ensure_minimum_size(image, min_size=1000):
    """
    Убеждается, что изображение имеет минимальный размер.
    Если изображение меньше min_size по любой стороне - масштабирует.

    === ВХОДНЫЕ ПАРАМЕТРЫ ===
    Args:
        image (PIL.Image): Входное изображение
        min_size (int): Минимальный размер стороны в пикселях (по умолчанию 1000)

    === ВОЗВРАЩАЕМЫЕ ЗНАЧЕНИЯ ===
    Returns:
        PIL.Image: Изображение с размером не менее min_size по каждой стороне

    === ВНУТРЕННИЕ ПАРАМЕТРЫ ===
        min_size (int): 1000 - минимальный размер
            - Увеличение: более высокое разрешение, но медленнее
            - Уменьшение: более низкое разрешение, но быстрее

        Resampling.LANCZOS: метод ресемплинга (высокое качество)
            - Используется при увеличении изображения

        scale (float): max(min_size / w, min_size / h) - коэффициент масштабирования
            - Выбирает максимальный коэффициент, чтобы обе стороны стали >= min_size

    === ПРИМЕР ===
        Изображение 621x912 с min_size=1000:
        - scale = max(1000/621, 1000/912) = max(1.61, 1.10) = 1.61
        - new_w = 621 * 1.61 = 1000
        - new_h = 912 * 1.61 = 1468
        - Результат: 1000x1468
    """
    w, h = image.size
    print(f"  Оригинальный размер: {w}x{h}")

    # Если изображение слишком маленькое - увеличиваем
    if w < min_size or h < min_size:
        # Вычисляем коэффициент масштабирования
        scale = max(min_size / w, min_size / h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        print(f"  Увеличиваем до: {new_w}x{new_h} (коэффициент {scale:.2f})")
        image = image.resize((new_w, new_h), Image.Resampling.LANCZOS)

    return image


def process_wrapper(file_input):
    """
    Обработчик загруженного файла для Gradio интерфейса.
    Определяет тип входных данных, загружает изображение и вызывает основную функцию.

    === ВХОДНЫЕ ПАРАМЕТРЫ ===
    Args:
        file_input: Входные данные от Gradio (может быть разных типов)
            - dict: {'path': '...'} или {'data': '...'} (base64)
            - bytes: PDF файл
            - str: путь к файлу
            - PIL.Image: изображение
            - None: ничего не загружено

    === ВОЗВРАЩАЕМЫЕ ЗНАЧЕНИЯ ===
    Returns:
        Tuple[PIL.Image, PIL.Image] | Tuple[None, None]:
            - (original_image, result_image) при успехе
            - (None, None) при ошибке или отсутствии данных

    === ВНУТРЕННИЕ ПАРАМЕТРЫ ===
        filename (str): "uploaded_sketch.png" - имя файла по умолчанию
        TEMP_PDF_RENDER (str): "/tmp/gradio_pdf_render.png" - временный файл для PDF
        min_size (int): 1000 - минимальный размер изображения

    === ПОДДЕРЖИВАЕМЫЕ ТИПЫ ===
        1. dict с 'path' - загрузка из файловой системы
        2. dict с 'data' - загрузка из base64 (вставка изображения)
        3. bytes - загрузка PDF файла
        4. str - путь к файлу
        5. PIL.Image - готовое изображение

    === ПРИМЕР ВЫЗОВА ===
        # Gradio автоматически вызывает эту функцию при загрузке
        result = process_wrapper(uploaded_file)
    """
    print(f"\n=== process_wrapper вызван ===")
    print(f"Тип file_input: {type(file_input)}")

    if file_input is None:
        print("file_input is None")
        return None, None

    img_obj = None
    filename = "uploaded_sketch.png"

    # === ОБРАБОТКА СЛОВАРЯ (от Gradio) ===
    if isinstance(file_input, dict):
        print("Это словарь, содержимое:", file_input.keys())

        # Вариант 1: путь к файлу
        if 'path' in file_input:
            print(f"Путь к файлу: {file_input['path']}")
            try:
                img_obj = Image.open(file_input['path']).convert("RGB")
                filename = os.path.basename(file_input['path'])
                print(f"Изображение загружено из пути: {filename}")
            except Exception as e:
                print(f"Ошибка открытия по пути: {e}")
                return None, None

        # Вариант 2: данные base64
        elif 'data' in file_input:
            print("Получены данные base64")
            import base64
            from io import BytesIO
            try:
                img_data = base64.b64decode(file_input['data'])
                img_obj = Image.open(BytesIO(img_data)).convert("RGB")
                filename = "uploaded_image.png"
                print("Изображение загружено из base64")
            except Exception as e:
                print(f"Ошибка декодирования base64: {e}")
                return None, None

    # === ОБРАБОТКА BYTES (PDF) ===
    elif isinstance(file_input, bytes):
        print("Это bytes (PDF файл)")
        temp_upload_path = "/tmp/user_uploaded.pdf"
        with open(temp_upload_path, "wb") as f:
            f.write(file_input)
        img_obj = render_pdf_to_image(temp_upload_path)
        filename = "uploaded_pdf_page1.png"
        print(f"PDF обработан, img_obj: {img_obj is not None}")

    # === ОБРАБОТКА СТРОКИ (путь к файлу) ===
    elif isinstance(file_input, str):
        print(f"Это строка: {file_input[:100] if len(file_input) > 100 else file_input}")
        try:
            img_obj = Image.open(file_input).convert("RGB")
            p = Path(file_input)
            if p.exists():
                filename = p.name
            else:
                filename = "direct_upload.png"
            print(f"Изображение загружено из строки: {filename}")
        except Exception as e:
            print(f"Не удалось открыть файл по пути {file_input}: {e}")
            return None, None

    # === ОБРАБОТКА PIL IMAGE ===
    elif hasattr(file_input, 'mode'):
        print(f"Это PIL Image, mode={file_input.mode}, size={file_input.size}")
        img_obj = file_input
        if hasattr(file_input, 'filename'):
            filename = os.path.basename(file_input.filename)
            print(f"Имя файла: {filename}")

    else:
        print(f"Неизвестный тип: {type(file_input)}")
        return None, None

    if img_obj is None:
        print("img_obj is None, возвращаем None")
        return None, None

    # === МАСШТАБИРОВАНИЕ ДЛЯ МАЛЕНЬКИХ ИЗОБРАЖЕНИЙ ===
    img_obj = ensure_minimum_size(img_obj, min_size=1000)
    print(f"Размер после масштабирования: {img_obj.size}")

    # === ВЫЗОВ ОСНОВНОЙ ФУНКЦИИ ===
    print(f"Вызываем extract_part_contours с filename={filename}")
    result = extract_part_contours(img_obj, filename)
    print("=== process_wrapper завершен ===\n")
    return result


# === GRADIO ИНТЕРФЕЙС ===
with gr.Blocks() as demo:
    """
    Gradio веб-интерфейс для загрузки чертежей и отображения результатов.

    === КОМПОНЕНТЫ ===
        1. input_img: gr.Image - поле для загрузки изображения
           - type="pil": возвращает PIL Image
           - height=400: высота отображения
           - interactive=True: разрешает загрузку и перетаскивание

        2. result_img: gr.Image - поле для отображения результата
           - type="pil": принимает PIL Image
           - height=400: высота отображения

        3. run_btn: gr.Button - кнопка "Найти контур"
           - variant="primary": синий цвет
           - size="lg": большой размер

    === ПРИМЕРЫ ===
        gr.Examples - показывает примеры чертежей из DATA_DIR
        - Загружает первые 10 PNG из папки alpha_version_png
        - При клике автоматически загружает пример и обрабатывает

    === ОБРАБОТЧИКИ ===
        run_btn.click(): обработка при нажатии кнопки
        - Вызывает process_wrapper
        - Вход: input_img
        - Выход: [input_img, result_img] (обновляет оба поля)

    === ЗАПУСК ===
        demo.launch(debug=True): запуск с режимом отладки
        - debug=True: показывает ошибки в консоли
        - share=True: создает публичную ссылку (если добавить)
    """
    gr.Markdown("# Детекция контура детали\nПоддерживаемые форматы: перетащите сюда ваш **.png**, **.jpg** или **.pdf**.")

    with gr.Row():
        input_img = gr.Image(
            type="pil",
            label="Исходный чертеж",
            height=400,
            interactive=True
        )
        result_img = gr.Image(type="pil", label="Результат", height=400)

    def get_examples_list():
        """
        Получение списка примеров для быстрой загрузки.

        Returns:
            List[List[str]]: Список путей к PNG-файлам для отображения в примерах
        """
        examples = []
        if DATA_DIR.exists():
            png_files = sorted(DATA_DIR.glob("*.png"))[:10]  # Первые 10 файлов
            examples = [[str(f)] for f in png_files]
        return examples

    examples_list = get_examples_list()
    if examples_list:
        gr.Examples(
            examples=examples_list,
            inputs=[input_img],
            outputs=[result_img],
            fn=lambda x: process_wrapper(x),
            cache_examples=False
        )

    run_btn = gr.Button("Найти контур", variant="primary", size="lg")
    run_btn.click(fn=process_wrapper, inputs=input_img, outputs=[input_img, result_img])

demo.launch(debug=True)